# NBA 3.1 Model-Based Feature Selection (Section 5)

Chooses the final features with LightGBM for one **segment** (`SEGMENT` widget): `all` (every HCP), `writers` (existing writers) or `non_writers`. One job runs the three segments in parallel, then a comparison task.

**Input features:** the statistical clean-up list of the segment (`statistical_v2/selected_features_<group>.csv`: Claims, Patient events and Rx market features kept after stages 1-4, NBA 3.0 features protected, plus every promotional, static, Veeva, market access and calendar feature, which skip the clean-up). For `all`, the union of the writer and non-writer lists (a feature kept for both appears once).

**Rows:** training months c1-c24 of the sample `all_hcps_5_train_months` with its weights; test months c25-c26 only for evaluation. For `non_writers` and `all`, every training row with an NBRx is kept plus a random 5% of the rows without (`NON_NBRX_SHARE`), weighted up by 20 so the totals stay the same; `writers` uses every row.

**Method (agreed 2026-11-27):**
1. Two loss functions: **Poisson** and **Tweedie** (the "2ED" in the call notes). The Tweedie variance power is chosen on fold 1 from 1.1, 1.3, 1.5 (the NBA 3.0 range) by the lowest validation Poisson deviance.
2. 5 folds split by doctor (a doctor is never in training and validation at the same time), early stopping on the validation fold.
3. Importance = mean absolute SHAP value on the validation rows (LightGBM's own contributions). Noise bar = the highest importance among 500 **shadow** features (copies of random candidates with their values shuffled across rows, so they cannot carry real signal).
4. A feature is **selected** if it beats the noise bar in at least 3 of the 5 folds. No cap on the list size.
5. Refit on all training rows with the selected features (rounds = mean best iteration x 1.1) and score the test months.

**Metrics on the test months** (HCPs ranked by predicted NBRx within each month): share of **new writers** (non-writer HCP-months with an NBRx) in the top 5 / 10 / 20%, share of **existing-writer NBRx** in the top 5 / 10 / 20%, overall NBRx capture, Poisson deviance and a decile table.

**Outputs:** `model_based/<SEGMENT>/` next to the statistical folders, and one MLflow run per segment with a child run per loss in `/Users/patricio.guledjian.ext@bayer.com/kerendia_nba_3_1_model_based_selection`. LightGBM uses internal names `f00000...` (15 real names contain `<`); `feature_name_map.csv` maps them back.
**Widgets:** `SEGMENT`, `RUN_NAME`, `SAMPLE_NAME`, `STAT_VERSION`, `MAX_FEATURES` (0 = all; N = test on N spread features), `NON_NBRX_SHARE`, `WRITE_OUTPUT`.

In [0]:
%pip install lightgbm==4.7.0
dbutils.library.restartPython()

In [0]:
import numpy as np, pandas as pd, json, os, time
import lightgbm as lgb
import mlflow
from sklearn.model_selection import GroupKFold
from pyspark.sql import functions as F

dbutils.widgets.dropdown("SEGMENT", "writers", ["all", "writers", "non_writers"])
dbutils.widgets.text("RUN_NAME", "combined_2026_11_26_full")
dbutils.widgets.text("SAMPLE_NAME", "all_hcps_5_train_months")
dbutils.widgets.text("STAT_VERSION", "statistical_v2")
dbutils.widgets.text("MAX_FEATURES", "0")
dbutils.widgets.text("NON_NBRX_SHARE", "0.05")
dbutils.widgets.dropdown("WRITE_OUTPUT", "false", ["true", "false"])
SEGMENT        = dbutils.widgets.get("SEGMENT").strip()
RUN_NAME       = dbutils.widgets.get("RUN_NAME").strip()
SAMPLE_NAME    = dbutils.widgets.get("SAMPLE_NAME").strip()
STAT_VERSION   = dbutils.widgets.get("STAT_VERSION").strip().strip("/")
MAX_FEATURES   = int(dbutils.widgets.get("MAX_FEATURES") or 0)
NON_NBRX_SHARE = float(dbutils.widgets.get("NON_NBRX_SHARE") or 0.05)
WRITE_OUTPUT   = dbutils.widgets.get("WRITE_OUTPUT") == "true"
assert SEGMENT in ("all", "writers", "non_writers") and 0 < NON_NBRX_SHARE <= 1

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"
COMBINED_DIR    = f"{DEV}{RUN_NAME}/hcp_combined/_sample_{SAMPLE_NAME}/"
COMBINED_PATH   = COMBINED_DIR + "kerendia_nba_3_1_hcp_selected_features_data"
DICTIONARY_PATH = COMBINED_DIR + "kerendia_nba_3_1_hcp_feature_dictionary"
TARGET_PATH     = VOL31 + "notebook_output/target_variable/target_variable"
FS_DIR   = f"{DEV}{RUN_NAME}/feature_selection/_sample_{SAMPLE_NAME}/"
STAT_DIR = f"{FS_DIR}{STAT_VERSION}/"
IS_TEST  = bool(MAX_FEATURES) or "_test_" in STAT_VERSION   # test runs never write into the real output folder
OUT_DIR  = f"{FS_DIR}model_based/" + (f"_test_{MAX_FEATURES}_features/" if IS_TEST else "") + f"{SEGMENT}/"
assert OUT_DIR.startswith(VOL31) and "kerendia_nba_3_volume" not in OUT_DIR
EXPERIMENT = "/Users/patricio.guledjian.ext@bayer.com/kerendia_nba_3_1_model_based_selection"

SEGMENT_GROUPS = {"all": ["writers", "non_writers"], "writers": ["writers"], "non_writers": ["non_writers"]}
WRITER_FLAG    = {"writers": True, "non_writers": False}
LOSSES         = ["poisson", "tweedie"]
TWEEDIE_POWERS = [1.1, 1.3, 1.5]   # the NBA 3.0 search range was 1.1-1.5
N_FOLDS, MIN_FOLDS_BEAT = 5, 3     # selected = beats the shadow bar in at least 3 of 5 folds
N_SHADOW   = 500                   # shuffled copies of random candidates = noise bar
SHAP_ROWS  = 10000                 # validation rows used for SHAP per fold
TOP_SHARES = [0.05, 0.10, 0.20]    # top shares of the ranking for the capture metrics
LOAD_CHUNK = 500                   # columns per Spark read
MAX_ROUNDS, EARLY_STOP, REFIT_ROUNDS_FACTOR = 3000, 100, 1.1
SEED = 42

def usable_cores():
    """Cores this task may really use: the smaller of the CPU affinity and the container CPU quota.
    os.cpu_count() can report the host's cores; more LightGBM threads than cores makes training many times slower."""
    n = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
    try:
        quota, period = open("/sys/fs/cgroup/cpu.max").read().split()
        if quota != "max":
            n = min(n, max(1, int(int(quota) // int(period))))
    except Exception:
        pass
    return max(1, n)
N_THREADS = usable_cores()
print(f"cores: os.cpu_count()={os.cpu_count()}, usable={N_THREADS}")
# Fixed parameters inside the NBA 3.0 search ranges (no tuning in the selection step)
BASE_PARAMS = dict(learning_rate=0.05, num_leaves=63, max_depth=7, min_child_samples=100, bagging_fraction=0.8,
                   bagging_freq=1, feature_fraction=0.5, lambda_l1=1.0, lambda_l2=1.0, max_bin=255, force_col_wise=True,
                   verbosity=-1, seed=SEED, num_threads=N_THREADS)
KEY_COLS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "SPLIT", "IS_EXISTING_WRITER", "WEIGHT"]
print(f"SEGMENT={SEGMENT}  STAT_VERSION={STAT_VERSION}  MAX_FEATURES={MAX_FEATURES}  NON_NBRX_SHARE={NON_NBRX_SHARE}  WRITE_OUTPUT={WRITE_OUTPUT}")
print("Lists :", STAT_DIR)
print("Output:", OUT_DIR)

In [0]:
combined   = spark.read.format("delta").load(COMBINED_PATH)
dictionary = spark.read.format("delta").load(DICTIONARY_PATH).toPandas().set_index("FEATURE_NAME")
lists = {g: pd.read_csv(f"{STAT_DIR}selected_features_{g}.csv") for g in SEGMENT_GROUPS[SEGMENT]}
for g, l in lists.items():
    assert l.FEATURE.is_unique, f"{g}: repeated features in the statistical list"
# Union of the lists: a feature kept for both groups appears once
in_list = pd.DataFrame({g: pd.Series(True, index=l.FEATURE.values) for g, l in lists.items()}).fillna(False).astype(bool)
CANDIDATES = sorted(in_list.index)
assert set(CANDIDATES) <= set(combined.columns) and set(CANDIDATES) <= set(dictionary.index), "list feature not in the table"
PROTECTED_3_0 = set().union(*[set(l.FEATURE[l.PROTECTED.astype(bool)]) for l in lists.values()])
if MAX_FEATURES:
    CANDIDATES = CANDIDATES[::max(1, len(CANDIDATES) // MAX_FEATURES)][:MAX_FEATURES]
CPOS    = {f: j for j, f in enumerate(CANDIDATES)}
FEAT_ID = {f: f"f{j:05d}" for j, f in enumerate(CANDIDATES)}   # LightGBM names (some real names contain '<')
P = len(CANDIDATES)
print({g: len(l) for g, l in lists.items()}, f"-> {P:,} candidate features for {SEGMENT}"
      + (f" (in both lists: {int(in_list.all(axis=1).sum()):,})" if len(lists) > 1 else ""))
display(pd.Series(dictionary.CATEGORY.reindex(CANDIDATES).values).value_counts().rename_axis("CATEGORY").reset_index(name="candidates"))

In [0]:
target = (spark.read.format("delta").load(TARGET_PATH)
          .select("HCP_COHORT_ID", F.col("KERENDIA_NBRX").cast("double").alias("TARGET_NBRX")))
keys = (combined.select(*KEY_COLS).join(target, "HCP_COHORT_ID", "left").toPandas()
        .sort_values("HCP_COHORT_ID").reset_index(drop=True))   # sorted so the random draw below is repeatable
keys.attrs = {}   # toPandas() puts Spark query metrics in attrs; they break parquet writing
assert keys.HCP_COHORT_ID.is_unique and keys.TARGET_NBRX.notna().all(), "duplicate rows or rows without a target"
keys = keys[keys.IS_EXISTING_WRITER.isin([WRITER_FLAG[g] for g in SEGMENT_GROUPS[SEGMENT]])].reset_index(drop=True)

rng = np.random.default_rng(SEED)
tr = keys[keys.SPLIT == "train"].reset_index(drop=True)
has_nbrx = tr.TARGET_NBRX.values > 0
if SEGMENT == "writers":
    keep = np.ones(len(tr), dtype=bool)
    tr["W"] = tr.WEIGHT
else:   # every row with an NBRx + a random share of the others, weighted up so the totals stay the same
    keep = has_nbrx | (rng.random(len(tr)) < NON_NBRX_SHARE)
    tr["W"] = tr.WEIGHT * np.where(has_nbrx, 1.0, 1.0 / NON_NBRX_SHARE)
TRAIN = tr[keep].reset_index(drop=True)
TEST  = keys[keys.SPLIT == "test"].reset_index(drop=True)

def describe(df, name, wcol):
    return {"rows": name, "n": len(df), "doctors": df.PROVIDER_ID.nunique(), "rows_with_nbrx": int((df.TARGET_NBRX > 0).sum()),
            "nbrx": float(df.TARGET_NBRX.sum()), "weighted_rows": float(df[wcol].sum()),
            "weighted_nbrx": float((df[wcol] * df.TARGET_NBRX).sum()), "writer_rows": int(df.IS_EXISTING_WRITER.sum())}
rows_summary = pd.DataFrame([describe(tr, "train, all rows", "WEIGHT"), describe(TRAIN, "train, used", "W"),
                             describe(TEST, "test c25-c26", "WEIGHT")])
assert np.isclose(rows_summary.weighted_nbrx[0], rows_summary.weighted_nbrx[1]), "the sample changed the weighted NBRx total"
print(f"weighted rows: all {rows_summary.weighted_rows[0]:,.0f} vs used {rows_summary.weighted_rows[1]:,.0f} "
      f"({rows_summary.weighted_rows[1] / rows_summary.weighted_rows[0] - 1:+.2%})")
display(rows_summary)

In [0]:
def load_matrix(rows, cols, extra=0):
    """Float32 matrix of `cols` for `rows` (same order), read LOAD_CHUNK columns at a time; `extra` empty columns at the end."""
    pos = spark.createDataFrame(pd.DataFrame({"HCP_COHORT_ID": rows.HCP_COHORT_ID.values,
                                              "_POS": np.arange(len(rows), dtype=np.int32)}))
    src = combined.join(F.broadcast(pos), "HCP_COHORT_ID")
    X = np.empty((len(rows), len(cols) + extra), dtype=np.float32)
    t0 = time.time()
    for s in range(0, len(cols), LOAD_CHUNK):
        cs = cols[s:s + LOAD_CHUNK]
        pdf = src.select("_POS", *[F.col(f"`{c}`") for c in cs]).toPandas()
        assert len(pdf) == len(rows) and pdf["_POS"].nunique() == len(rows), "rows missing or repeated in the combined table"
        X[pdf["_POS"].to_numpy(), s:s + len(cs)] = pdf[cs].to_numpy(dtype=np.float32)
        del pdf
        print(f"  loaded {s + len(cs):,}/{len(cols):,} columns for {len(rows):,} rows ({(time.time() - t0) / 60:.1f} min)")
    assert not np.isnan(X[:, :len(cols)]).any(), "empty values in the combined table"
    return X

n_shadow = min(N_SHADOW, P)
XA = load_matrix(TRAIN, CANDIDATES, extra=n_shadow)
y, w, groups = TRAIN.TARGET_NBRX.values, TRAIN.W.values, TRAIN.PROVIDER_ID.values
# Shadow features: copies of random candidates with their values shuffled across rows (no link to NBRx left)
shadow_of = np.sort(rng.choice(P, size=n_shadow, replace=False))
for i, j in enumerate(shadow_of):
    XA[:, P + i] = XA[rng.permutation(len(XA)), j]
NAMES     = CANDIDATES + [f"SHADOW_{i:03d}_{CANDIDATES[j]}" for i, j in enumerate(shadow_of)]
LGB_NAMES = [FEAT_ID[f] for f in CANDIDATES] + [f"s{i:03d}" for i in range(n_shadow)]
DS_PARAMS = {"max_bin": BASE_PARAMS["max_bin"], "verbosity": -1, "seed": SEED}
DS = lgb.Dataset(XA, label=y, weight=w, feature_name=LGB_NAMES, free_raw_data=False, params=DS_PARAMS)
DS.construct()

# 5 folds split by doctor: all months of a doctor are in the same fold
FOLDS = [(np.sort(a), np.sort(b)) for a, b in
         GroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y, groups)]
for a, b in FOLDS:
    assert not set(groups[a]) & set(groups[b]), "a doctor is in training and validation"
display(pd.DataFrame([{"fold": k + 1, "train_rows": len(a), "val_rows": len(b), "val_doctors": len(set(groups[b])),
                       "val_nbrx": float(y[b].sum())} for k, (a, b) in enumerate(FOLDS)]))
print(f"training matrix {XA.shape[0]:,} rows x {XA.shape[1]:,} columns ({P:,} candidates + {n_shadow} shadows), "
      f"{XA.nbytes / 1e9:.1f} GB")

In [0]:
def poisson_deviance(yt, mu, wt):
    """Weighted mean Poisson deviance (same scale for both losses)."""
    mu = np.clip(mu, 1e-9, None)
    d = 2 * (np.where(yt > 0, yt * np.log(np.where(yt > 0, yt, 1) / mu), 0) - (yt - mu))
    return float((wt * d).sum() / wt.sum())

def loss_params(loss, power=None):
    p = dict(BASE_PARAMS, objective=loss, metric=loss)
    if loss == "tweedie":
        p["tweedie_variance_power"] = power
    return p

def fit_fold(params, k):
    a, b = FOLDS[k]
    t0 = time.time()
    dtr, dva = DS.subset(a), DS.subset(b)
    dtr.construct(); dva.construct()
    t1 = time.time()
    bst = lgb.train(params, dtr, MAX_ROUNDS, valid_sets=[dva], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    t2 = time.time()
    pred = bst.predict(XA[b], num_iteration=bst.best_iteration)
    print(f"    fold {k + 1} {params['objective']}: setup {t1 - t0:.0f}s, training {t2 - t1:.0f}s "
          f"({bst.current_iteration()} rounds, {(t2 - t1) / max(1, bst.current_iteration()):.2f}s per round), predict {time.time() - t2:.0f}s")
    return bst, pred

def shap_importance(bst, idx):
    sub = idx if len(idx) <= SHAP_ROWS else np.sort(rng.choice(idx, SHAP_ROWS, replace=False))
    contrib = bst.predict(XA[sub], num_iteration=bst.best_iteration, pred_contrib=True)
    return np.abs(contrib[:, :-1]).mean(axis=0)   # last column is the base value

t_cv = time.time()
# Tweedie variance power on fold 1, compared on Poisson deviance (same scale for every power)
power_rows, power_fits = [], {}
for pw in TWEEDIE_POWERS:
    bst, pred = fit_fold(loss_params("tweedie", pw), 0)
    b = FOLDS[0][1]
    power_rows.append({"tweedie_variance_power": pw, "best_iteration": bst.best_iteration,
                       "val_poisson_deviance": poisson_deviance(y[b], pred, w[b])})
    power_fits[pw] = (bst, pred)
    print(f"tweedie power {pw}: {power_rows[-1]}  ({(time.time() - t_cv) / 60:.1f} min)")
power_table = pd.DataFrame(power_rows)
BEST_POWER = float(power_table.sort_values("val_poisson_deviance").tweedie_variance_power.iloc[0])
print("chosen tweedie variance power:", BEST_POWER)

fold_rows, IMP = [], {}
for loss in LOSSES:
    imps = np.zeros((N_FOLDS, len(NAMES)))
    for k in range(N_FOLDS):
        if loss == "tweedie" and k == 0:
            bst, pred = power_fits[BEST_POWER]          # already trained in the power search
        else:
            bst, pred = fit_fold(loss_params(loss, BEST_POWER), k)
        b = FOLDS[k][1]
        imps[k] = shap_importance(bst, b)
        bar = imps[k, P:].max()
        fold_rows.append({"loss": loss, "fold": k + 1, "best_iteration": bst.best_iteration, "val_rows": len(b),
                          "val_nbrx": float(y[b].sum()), "val_poisson_deviance": poisson_deviance(y[b], pred, w[b]),
                          "shadow_bar": float(bar), "features_beating_bar": int((imps[k, :P] > bar).sum()),
                          "minutes_so_far": round((time.time() - t_cv) / 60, 1)})
        print(fold_rows[-1])
    IMP[loss] = imps
del power_fits
folds_table = pd.DataFrame(fold_rows)

SELECTION = {}
for loss in LOSSES:
    imps = IMP[loss]
    bars = imps[:, P:].max(axis=1)
    t = pd.DataFrame({"FEATURE": CANDIDATES, "CATEGORY": dictionary.CATEGORY.reindex(CANDIDATES).values,
                      "SUB_CATEGORY": dictionary.SUB_CATEGORY.reindex(CANDIDATES).values,
                      "LGB_NAME": [FEAT_ID[f] for f in CANDIDATES],
                      "FOLDS_BEATING_SHADOWS": (imps[:, :P] > bars[:, None]).sum(axis=0),
                      "MEAN_ABS_SHAP": imps[:, :P].mean(axis=0),
                      "MEAN_SHAP_VS_BAR": (imps[:, :P] / np.where(bars > 0, bars, np.nan)[:, None]).mean(axis=0),
                      "PROTECTED_3_0": [f in PROTECTED_3_0 for f in CANDIDATES]})
    for g in lists:
        t[f"IN_{g.upper()}_LIST"] = in_list[g].reindex(CANDIDATES).fillna(False).astype(bool).values
    t["SELECTED"] = t.FOLDS_BEATING_SHADOWS >= MIN_FOLDS_BEAT
    t = t.sort_values(["SELECTED", "MEAN_ABS_SHAP"], ascending=False).reset_index(drop=True)
    t["RANK"] = np.arange(1, len(t) + 1)
    SELECTION[loss] = t
    print(f"{loss}: {int(t.SELECTED.sum()):,} of {P:,} selected; protected 3.0 features selected "
          f"{int((t.SELECTED & t.PROTECTED_3_0).sum()):,} of {int(t.PROTECTED_3_0.sum()):,}")
display(folds_table)
display(pd.concat([SELECTION[l].assign(loss=l) for l in LOSSES]).groupby(["loss", "CATEGORY"])
        .agg(candidates=("FEATURE", "size"), selected=("SELECTED", "sum")).reset_index())

In [0]:
SELECTED = {l: [f for f in CANDIDATES if f in set(SELECTION[l].FEATURE[SELECTION[l].SELECTED])] for l in LOSSES}
assert all(SELECTED.values()), "a loss selected no feature"
TEST_COLS = [f for f in CANDIDATES if any(f in set(SELECTED[l]) for l in LOSSES)]
XT = load_matrix(TEST, TEST_COLS)
TPOS = {f: j for j, f in enumerate(TEST_COLS)}

MODELS, refit_rows = {}, []
PRED = TEST[["HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "IS_EXISTING_WRITER", "TARGET_NBRX"]].copy()
for loss in LOSSES:
    sel = SELECTED[loss]
    rounds = max(1, int(round(folds_table[folds_table.loss == loss].best_iteration.mean() * REFIT_ROUNDS_FACTOR)))
    d = lgb.Dataset(XA[:, [CPOS[f] for f in sel]], label=y, weight=w, feature_name=[FEAT_ID[f] for f in sel], params=DS_PARAMS)
    t0 = time.time()
    bst = lgb.train(loss_params(loss, BEST_POWER), d, rounds)
    del d
    MODELS[loss] = bst
    PRED[f"PRED_{loss.upper()}"] = bst.predict(XT[:, [TPOS[f] for f in sel]])
    refit_rows.append({"loss": loss, "features": len(sel), "rounds": rounds, "train_rows": len(y),
                       "minutes": round((time.time() - t0) / 60, 1),
                       "test_poisson_deviance": poisson_deviance(PRED.TARGET_NBRX.values, PRED[f"PRED_{loss.upper()}"].values,
                                                                 np.ones(len(PRED)))})
refit_table = pd.DataFrame(refit_rows)

def capture(df, pred_col):
    """HCPs ranked by prediction within each test month; shares of new writers / writer NBRx / all NBRx in the top."""
    writer = df.IS_EXISTING_WRITER.astype(bool).values
    new_writer = ~writer & (df.TARGET_NBRX.values > 0)
    yv = df.TARGET_NBRX.values
    rank = df.groupby("COHORT_ID")[pred_col].rank(ascending=False, method="first").values
    size = df.groupby("COHORT_ID")[pred_col].transform("size").values
    out = []
    for s in TOP_SHARES:
        top = rank <= np.ceil(size * s)
        out.append({"pred": pred_col, "top_share": s, "rows_in_top": int(top.sum()),
                    "new_writers": int(new_writer.sum()), "new_writers_in_top": int((new_writer & top).sum()),
                    "new_writer_capture": (new_writer & top).sum() / new_writer.sum() if new_writer.any() else np.nan,
                    "writer_nbrx": float(yv[writer].sum()), "writer_nbrx_in_top": float(yv[writer & top].sum()),
                    "writer_nbrx_capture": yv[writer & top].sum() / yv[writer].sum() if yv[writer].sum() > 0 else np.nan,
                    "nbrx_capture": yv[top].sum() / yv.sum()})
    return out

def deciles(df, pred_col):
    dec = df.groupby("COHORT_ID")[pred_col].rank(ascending=False, method="first", pct=True).mul(10).apply(np.ceil).clip(1, 10)
    g = df.assign(DECILE=dec.astype(int), NEW_WRITER=(~df.IS_EXISTING_WRITER.astype(bool)) & (df.TARGET_NBRX > 0))
    out = g.groupby("DECILE").agg(rows=("TARGET_NBRX", "size"), nbrx=("TARGET_NBRX", "sum"), new_writers=("NEW_WRITER", "sum"),
                                  mean_pred=(pred_col, "mean")).reset_index()
    out["nbrx_share"] = out.nbrx / out.nbrx.sum()
    return out.assign(pred=pred_col)

metrics_table = pd.DataFrame([r for l in LOSSES for r in capture(PRED, f"PRED_{l.upper()}")])
decile_table  = pd.concat([deciles(PRED, f"PRED_{l.upper()}") for l in LOSSES], ignore_index=True)
display(refit_table)
display(metrics_table)

In [0]:
for loss in LOSSES:
    t = SELECTION[loss]
    assert len(t) == P and t.FEATURE.is_unique and not t.FEATURE.str.startswith("SHADOW_").any()
    assert (t.SELECTED == (t.FOLDS_BEATING_SHADOWS >= MIN_FOLDS_BEAT)).all()
    p = PRED[f"PRED_{loss.upper()}"].values
    assert np.isfinite(p).all() and (p >= 0).all(), f"{loss}: invalid predictions"
    assert MODELS[loss].num_feature() == len(SELECTED[loss]), f"{loss}: model features differ from the selected list"
# Each test month ranked separately; recompute the top-10% capture another way (sort instead of rank)
for loss in LOSSES:
    col, got = f"PRED_{loss.upper()}", metrics_table.set_index(["pred", "top_share"])
    top = []
    for _, g in PRED.groupby("COHORT_ID"):
        top.append(g.sort_values(col, ascending=False, kind="mergesort").head(int(np.ceil(len(g) * 0.10))))
    top = pd.concat(top)
    assert np.isclose(top.TARGET_NBRX.sum() / PRED.TARGET_NBRX.sum(), got.loc[(col, 0.10), "nbrx_capture"]), f"{loss}: capture check"
assert len(PRED) == len(TEST) and PRED.HCP_COHORT_ID.is_unique
assert set(PRED.COHORT_ID) == set(TEST.COHORT_ID)
print("checks passed:", {l: len(SELECTED[l]) for l in LOSSES}, "selected; test months", sorted(PRED.COHORT_ID.unique()))

In [0]:
summary = {"segment": SEGMENT, "run_name": RUN_NAME, "sample_name": SAMPLE_NAME, "stat_version": STAT_VERSION,
           "is_test": IS_TEST, "max_features": MAX_FEATURES, "non_nbrx_share": NON_NBRX_SHARE, "candidates": P,
           "shadows": n_shadow, "train_rows": len(TRAIN), "test_rows": len(TEST), "best_tweedie_power": BEST_POWER,
           "selected": {l: len(SELECTED[l]) for l in LOSSES}, "base_params": BASE_PARAMS,
           "minutes": round((time.time() - t_cv) / 60, 1)}
if WRITE_OUTPUT:
    os.makedirs(OUT_DIR, exist_ok=True)
    for loss in LOSSES:
        SELECTION[loss].to_csv(f"{OUT_DIR}feature_importance_{loss}.csv", index=False)
        SELECTION[loss][SELECTION[loss].SELECTED].to_csv(f"{OUT_DIR}selected_features_{loss}.csv", index=False)
    folds_table.to_csv(OUT_DIR + "cv_folds.csv", index=False)
    power_table.to_csv(OUT_DIR + "tweedie_power.csv", index=False)
    refit_table.to_csv(OUT_DIR + "refit.csv", index=False)
    metrics_table.to_csv(OUT_DIR + "test_metrics.csv", index=False)
    decile_table.to_csv(OUT_DIR + "test_deciles.csv", index=False)
    rows_summary.to_csv(OUT_DIR + "rows_summary.csv", index=False)
    pd.DataFrame({"LGB_NAME": LGB_NAMES, "FEATURE": NAMES}).to_csv(OUT_DIR + "feature_name_map.csv", index=False)
    PRED.attrs = {}
    PRED.to_parquet(OUT_DIR + "test_predictions.parquet", index=False)
    with open(OUT_DIR + "run_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2)

    mlflow.set_experiment(EXPERIMENT)
    with mlflow.start_run(run_name=SEGMENT + (" (test)" if IS_TEST else "")):
        mlflow.set_tags({"segment": SEGMENT, "stat_version": STAT_VERSION, "is_test": str(IS_TEST), "step": "model_based_selection"})
        mlflow.log_params({k: v for k, v in summary.items() if k not in ("selected", "base_params", "minutes")})
        for loss in LOSSES:
            with mlflow.start_run(run_name=f"{SEGMENT}_{loss}", nested=True):
                mlflow.set_tags({"segment": SEGMENT, "loss": loss})
                mlflow.log_params({**loss_params(loss, BEST_POWER), "n_selected": len(SELECTED[loss]),
                                   "refit_rounds": int(refit_table.set_index("loss").loc[loss, "rounds"])})
                f = folds_table[folds_table.loss == loss]
                mlflow.log_metrics({"cv_val_poisson_deviance": float(f.val_poisson_deviance.mean()),
                                    "test_poisson_deviance": float(refit_table.set_index("loss").loc[loss, "test_poisson_deviance"])})
                for _, r in metrics_table[metrics_table.pred == f"PRED_{loss.upper()}"].iterrows():
                    s = int(round(r.top_share * 100))
                    for m in ["new_writer_capture", "writer_nbrx_capture", "nbrx_capture"]:
                        if pd.notna(r[m]):
                            mlflow.log_metric(f"test_{m}_top{s}", float(r[m]))
                sel = SELECTED[loss]
                ex = pd.DataFrame(XT[:5, [TPOS[c] for c in sel]], columns=[FEAT_ID[c] for c in sel])
                mlflow.lightgbm.log_model(MODELS[loss], name="model", input_example=ex)
                mlflow.log_artifact(f"{OUT_DIR}selected_features_{loss}.csv")
                mlflow.log_artifact(f"{OUT_DIR}feature_importance_{loss}.csv")
        for fn in ["cv_folds.csv", "tweedie_power.csv", "refit.csv", "test_metrics.csv", "test_deciles.csv",
                   "rows_summary.csv", "feature_name_map.csv", "run_summary.json"]:
            mlflow.log_artifact(OUT_DIR + fn)
    print("Saved to", OUT_DIR, sorted(os.listdir(OUT_DIR)))
else:
    print("WRITE_OUTPUT = false: nothing saved")
print(json.dumps(summary, indent=1, default=str))